In [ ]:
import numpy as np
import torch
import os
import nibabel as nib
from pathlib import Path
from tqdm import tqdm
from UNet.unet_model import UNet


In [ ]:
SOURCE_PATH = "/content"  

DATA_DIR = Path(SOURCE_PATH) / "data/nii_pre_unet"
MAPS_DIR = Path(SOURCE_PATH) / "data/maps_post_unet"
MAPS_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH = Path(SOURCE_PATH) / "checkpoints/trial-37_best.pth"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


In [ ]:
from xml.parsers.expat import model


LOW, HIGH = -100, 400

def windowing(img):
    img = np.clip(img, LOW, HIGH)
    return (img - LOW) / (HIGH - LOW)


def dice_3d(pred_xyz: np.ndarray, gt_xyz: np.ndarray, eps=1e-6) -> float:
    pred = pred_xyz > 0
    gt = gt_xyz > 0
    inter = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    if denom == 0:
        return np.nan
    return (2 * inter + eps) / (denom + eps)


def load_model_once():
    """Loads the model and handles the PytorchStreamReader error."""
    if not os.path.exists(CKPT_PATH):
        raise FileNotFoundError(f"Checkpoint not found at {CKPT_PATH}")

    print(f"Loading model from {CKPT_PATH}...")
    # change this requirding to the hyperparameters used during training
    model = UNet(1, 1, bilinear=False, dropout=0.2).to(DEVICE)

    try:
        ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state_dict"])
        model.eval()
        print("Model loaded successfully.")
        return model
    except Exception as e:
        print(f"ERROR LOADING MODEL: {e}")
        print("the checkpoint file is likely corrupted or incomplete.")
        return None



def infer_volume(index: int, model, threshold: float = 0.5):
    # Load CT
    vol_path = DATA_DIR / f"volume-{index}.nii"
    seg_path = DATA_DIR / f"segmentation-{index}.nii"

    if not vol_path.exists() or not seg_path.exists():
        raise FileNotFoundError(f"Missing files for index {index}")

    nii = nib.load(str(vol_path))
    vol = nii.get_fdata().astype(np.float32)
    X, Y, Z = vol.shape

    # Load GT
    gt = nib.load(str(seg_path)).get_fdata()
    gt = (gt == 2).astype(np.uint8)  # tumor only

    probas = np.zeros((Z, X, Y), dtype=np.float32)

    with torch.no_grad():
        for z in range(Z):
            sl = windowing(vol[:, :, z])
            x = torch.from_numpy(sl)[None, None].to(DEVICE)
            logits = model(x)
            proba = torch.sigmoid(logits)
            probas[z] = proba[0, 0].cpu().numpy()

    # Save probabilities
    np.save(MAPS_DIR / f"post_unet_{index}.npy", probas)

    # Dice volumique
    pred_xyz = (np.transpose(probas, (1, 2, 0)) >= threshold).astype(np.uint8)
    dice = dice_3d(pred_xyz, gt)

    return index, dice


def run_inference_and_collect(indices, model):
    results = []
    if model is None:
        return results

    for idx in tqdm(indices):
        try:
            i, d = infer_volume(idx, model)
            results.append((i, d))
            print(f" Volume {i} | Dice = {d:.4f}")
        except Exception as e:
            print(f" [ERROR] Volume {idx}: {e}")
    return results



def select_top_3(results):
    results = [(i, d) for i, d in results if not np.isnan(d)]
    results = sorted(results, key=lambda x: x[1])

    dices = np.array([d for _, d in results])
    mean_dice = dices.mean()

    best = results[-1]
    second_best = results[-2]
    mean_case = min(results, key=lambda x: abs(x[1] - mean_dice))

    print("\nSélection finale")
    print(f"Best        : idx={best[0]}, dice={best[1]:.4f}")
    print(f"Second best : idx={second_best[0]}, dice={second_best[1]:.4f}")
    print(f"Mean-close  : idx={mean_case[0]}, dice={mean_case[1]:.4f}")

    return {
        "best": best,
        "second_best": second_best,
        "mean_case": mean_case,
        "mean_dice": mean_dice,
    }


# Exemple : indices du test set
test_indices = [6, 13, 17, 28, 62, 70, 94, 130] #hardcoded 

results = run_inference_and_collect(test_indices)
selected_cases = select_top_3(results)
